## Semi-structured and Multi-modal RAG

Many documents contain a mixture of content types, including text, tables, and images.

Semi-structured data can be challenging for conventional RAG for at least two reasons:

* Text splitting may break up tables, corrupting the data in retrieval
* Embedding tables may pose challenges for semantic similarity search

And the information captured in images is typically lost.

With the emergence of multimodal LLMs, like [GPT4-V](https://openai.com/research/gpt-4v-system-card), it is worth considering how to utilize images in RAG:

`Option 1:`

* Use multimodal embeddings (such as [CLIP](https://openai.com/research/clip)) to embed images and text
* Retrieve both using similarity search
* Pass raw images and text chunks to a multimodal LLM for answer synthesis

`Option 2:`

* Use a multimodal LLM (such as [GPT4-V](https://openai.com/research/gpt-4v-system-card), [LLaVA](https://llava.hliu.cc/), or [FUYU-8b](https://www.adept.ai/blog/fuyu-8b)) to produce text summaries from images
* Embed and retrieve text
* Pass text chunks to an LLM for answer synthesis

`Option 3:`

* Use a multimodal LLM (such as [GPT4-V](https://openai.com/research/gpt-4v-system-card), [LLaVA](https://llava.hliu.cc/), or [FUYU-8b](https://www.adept.ai/blog/fuyu-8b)) to produce text summaries from images
* Embed and retrieve image summaries with a reference to the raw image
* Pass raw images and text chunks to a multimodal LLM for answer synthesis   

This cookbook show how we might tackle this :

* We will use [Unstructured](https://unstructured.io/) to parse images, text, and tables from documents (PDFs).
* We will use the [multi-vector retriever](https://python.langchain.com/docs/modules/data_connection/retrievers/multi_vector) to store raw tables, text, (optionally) images along with their summaries for retrieval.
* We will demonstrate `Option 2`, and will follow-up on the other approaches in future cookbooks.

![ss_mm_rag.png](attachment:9bbbcfe4-2b85-4e76-996a-ce8d1497d34e.png)

## Packages

In [ ]:
! pip install -U langchain langchain-community langchain-classic langchain-openai openai tiktoken chromadb unstructured[all-docs] pydantic lxml

## Data Loading

### Partition PDF tables, text, and images
  
* `LLaVA` Paper: https://arxiv.org/pdf/2304.08485.pdf
* Use [Unstructured](https://unstructured-io.github.io/unstructured/) to partition elements

In [ ]:
path = "/Users/rlm/Desktop/Papers/LLaVA/"

In [ ]:
from lxml import html
from pydantic import BaseModel
from typing import Any, Optional
from unstructured.partition.pdf import partition_pdf

# Get elements
raw_pdf_elements = partition_pdf("/content/LAVA.pdf",
                                 # Using pdf format to find embedded image blocks
                                 extract_images_in_pdf=True,
                                 # Use layout model (YOLOX) to get bounding boxes (for tables) and find titles
                                 # Titles are any sub-section of the document
                                 infer_table_structure=True,
                                 # Post processing to aggregate text once we have the title
                                 chunking_strategy="by_title",
                                 # Chunking params to aggregate text blocks
                                 # Attempt to create a new chunk 3800 chars
                                 # Attempt to keep chunks > 2000 chars
                                 # Hard max on chunks
                                 max_characters=4000,
                                 new_after_n_chars=3800,
                                 combine_text_under_n_chars=2000,
                                 image_output_dir_path="/content/figures/")

yolox_l0.05.onnx:   0%|          | 0.00/217M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

In [ ]:
# Create a dictionary to store counts of each type
category_counts = {}

for element in raw_pdf_elements:
    category = str(type(element))
    if category in category_counts:
        category_counts[category] += 1
    else:
        category_counts[category] = 1

# Unique_categories will have unique elements
unique_categories = set(category_counts.keys())
category_counts

{"<class 'unstructured.documents.elements.CompositeElement'>": 27}

In [ ]:
class Element(BaseModel):
    type: str
    text: Any

# Categorize by type
categorized_elements = []
for element in raw_pdf_elements:
    if "unstructured.documents.elements.Table" in str(type(element)):
        categorized_elements.append(Element(type="table", text=str(element)))
    elif "unstructured.documents.elements.CompositeElement" in str(type(element)):
        categorized_elements.append(Element(type="text", text=str(element)))

# Tables
table_elements = [e for e in categorized_elements if e.type == "table"]
print(len(table_elements))

# Text
text_elements = [e for e in categorized_elements if e.type == "text"]
print(len(text_elements))

0
27


## Multi-vector retriever

Use [multi-vector-retriever](https://python.langchain.com/docs/modules/data_connection/retrievers/multi_vector#summary).

Summaries are used to retrieve raw tables and / or raw chunks of text.

### Text and Table summaries

In [ ]:
from langchain_community.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

# Prompt
prompt_text="""You are an assistant tasked with summarizing tables and text. \
Give a concise summary of the table or text. Table or text chunk: {element} """
prompt = ChatPromptTemplate.from_template(prompt_text)

# Summary chain
model = ChatOpenAI(temperature=0,model="gpt-4")
summarize_chain = {"element": lambda x:x} | prompt | model | StrOutputParser()

Enter your OpenAI API key: ··········


/tmp/ipython-input-1842513169.py:13: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  model = ChatOpenAI(temperature=0,model="gpt-4")


In [ ]:
# Apply to text
texts = [i.text for i in text_elements]
# Reduced max_concurrency to 1 to avoid RateLimitError
text_summaries = summarize_chain.batch(texts, {"max_concurrency": 1})

In [ ]:
# Apply to tables
tables = [i.text for i in table_elements]
table_summaries = summarize_chain.batch(tables, {"max_concurrency": 1})

### Images

We will implement `Option 2` discussed above:

* Use a multimodal LLM ([LLaVA](https://llava.hliu.cc/)) to produce text summaries from images
* Embed and retrieve text
* Pass text chunks to an LLM for answer synthesis

#### Image summaries

We will use [LLaVA](https://github.com/haotian-liu/LLaVA/), an open source multimodal model.

We will use [llama.cpp](https://github.com/ggerganov/llama.cpp/pull/3436) to run LLaVA locally (e.g., on a Mac laptop):

* Clone [llama.cpp](https://github.com/ggerganov/llama.cpp)
* Download the LLaVA model: `mmproj-model-f16.gguf` and one of `ggml-model-[f16|q5_k|q4_k].gguf` from [LLaVA 7b repo](https://huggingface.co/mys/ggml_llava-v1.5-7b/tree/main)
* Build
```
mkdir build && cd build && cmake ..
cmake --build .
```
* Run inference across images:
```
/Users/rlm/Desktop/Code/llama.cpp/bin/llava -m ../models/llava-7b/ggml-model-q5_k.gguf --mmproj ../models/llava-7b/mmproj-model-f16.gguf --temp 0.1 -p "Describe the image in detail. Be specific about graphs, such as bar plots." --image "$img" > "$output_file"
```

In [ ]:
%%bash

# Define paths
IMG_DIR="/content/figures/"
LLAMA_CPP_DIR="./llama.cpp"
MODEL_DIR="./models"

# 1. Clone and Build llama.cpp if not present
if [ ! -d "$LLAMA_CPP_DIR" ]; then
    echo "Cloning llama.cpp..."
    git clone https://github.com/ggerganov/llama.cpp "$LLAMA_CPP_DIR"
    echo "Building llama-llava-cli..."
    cd "$LLAMA_CPP_DIR"
    cmake -B build
    cmake --build build --config Release
    cd ..
fi

# 2. Download LLaVA models if not present
mkdir -p "$MODEL_DIR"
if [ ! -f "$MODEL_DIR/ggml-model-q5_k.gguf" ]; then
    echo "Downloading LLaVA models..."
    wget -q -O "$MODEL_DIR/ggml-model-q5_k.gguf" https://huggingface.co/mys/ggml_llava-v1.5-7b/resolve/main/ggml-model-q5_k.gguf
    wget -q -O "$MODEL_DIR/mmproj-model-f16.gguf" https://huggingface.co/mys/ggml_llava-v1.5-7b/resolve/main/mmproj-model-f16.gguf
fi

# 3. Run Inference
echo "Processing images..."
for img in "${IMG_DIR}"*.jpg; do
    # Check if file exists
    if [ -e "$img" ]; then
        base_name=$(basename "$img" .jpg)
        output_file="${IMG_DIR}${base_name}.txt"

        echo "Generating summary for $base_name..."

        # Run LLaVA inference
        # Suppress extensive logging with 2>/dev/null to keep output clean
        # Note: binary location might be in build/bin/ depending on cmake config, usually bin/llama-llava-cli
        "$LLAMA_CPP_DIR/build/bin/llama-llava-cli" \
            -m "$MODEL_DIR/ggml-model-q5_k.gguf" \
            --mmproj "$MODEL_DIR/mmproj-model-f16.gguf" \
            --temp 0.1 \
            -p "Describe the image in detail. Be specific about graphs, such as bar plots." \
            --image "$img" > "$output_file" 2>/dev/null
    fi
done
echo "Finished processing images."

Cloning llama.cpp...
Building llama-llava-cli...
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU bac

Cloning into './llama.cpp'...
CMAKE_BUILD_TYPE=Release


Note:

To run LLaVA with python bindings, we need a Python API to run the CLIP model.

CLIP support is likely to be added to `llama.cpp` in the future.

After running the above, we  fetch and clean image summaries.

In [ ]:
import os, glob

# Define the path to the images/summaries
path = "/content/figures/"

# Get all .txt file summaries
file_paths = glob.glob(os.path.join(path, "*.txt"))

# Read each file and store its content in a list
img_summaries = []
for file_path in file_paths:
    with open(file_path, 'r') as file:
        img_summaries.append(file.read()) # Keep the full content

# Clean up the summaries
# The logging header might vary or not be present depending on the LLaVA run.
# We'll try to split by a common header if it exists, otherwise just take the text.
cleaned_img_summary = []
for s in img_summaries:
    if "clip_model_load:" in s:
        cleaned_img_summary.append(s.split("clip_model_load:", 1)[1].split("\n\n", 1)[1].strip())
    else:
        cleaned_img_summary.append(s.strip())

### Add to vectorstore

Use [Multi Vector Retriever](https://python.langchain.com/docs/modules/data_connection/retrievers/multi_vector#summary) with summaries.

In [ ]:
import uuid
from langchain_community.vectorstores import Chroma
from langchain_core.stores import InMemoryStore
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever

# The vectorstore to use to index the child chunks
vectorstore = Chroma(
    collection_name="summaries",
    embedding_function=OpenAIEmbeddings()
)

# The storage layer for the parent documents
store = InMemoryStore()
id_key = "doc_id"

# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)

# Add texts
doc_ids = [str(uuid.uuid4()) for _ in texts]
summary_texts = [Document(page_content=s,metadata={id_key: doc_ids[i]}) for i, s in enumerate(text_summaries)]
if summary_texts:
    retriever.vectorstore.add_documents(summary_texts)
    retriever.docstore.mset(list(zip(doc_ids, texts)))
else:
    print("No text summaries to add to the vectorstore.")

# Add tables
table_ids = [str(uuid.uuid4()) for _ in tables]
summary_tables = [Document(page_content=s,metadata={id_key: table_ids[i]}) for i, s in enumerate(table_summaries)]
if summary_tables:
    retriever.vectorstore.add_documents(summary_tables)
    retriever.docstore.mset(list(zip(table_ids, tables)))
else:
    print("No table summaries to add to the vectorstore.")

/tmp/ipython-input-2451164708.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


No table summaries to add to the vectorstore.


For `option 2` (above):

* Store the image summary in the `docstore`, which we return to the LLM for answer generation.

In [ ]:
# Add image summaries
img_ids = [str(uuid.uuid4()) for _ in cleaned_img_summary]
summary_img = [Document(page_content=s,metadata={id_key: img_ids[i]}) for i, s in enumerate(cleaned_img_summary)]
retriever.vectorstore.add_documents(summary_img)
retriever.docstore.mset(list(zip(img_ids, cleaned_img_summary)))

For `option 3` (above):

* Store the images in the `docstore`.
* Using the image in answer synthesis will require a multimodal LLM with Python API integration.
* GPT4-V is expected soon, and - as mentioned above - CLIP support is likely to be added to `llama.cpp` in the future.

In [ ]:
# Add images
img_ids = [str(uuid.uuid4()) for _ in cleaned_img_summary]
summary_img = [Document(page_content=s,metadata={id_key: img_ids[i]}) for i, s in enumerate(cleaned_img_summary)]
retriever.vectorstore.add_documents(summary_img)
### Fetch images
# retriever.docstore.mset(list(zip(img_ids, ### image ### )))

['ec9d48c4-316c-4348-877c-51c47465d99e',
 '9b030c55-0c4d-44dd-a7fd-02630c22c9a1',
 'fd35c1a9-fec3-4e05-a29d-83e97e5f0ada',
 '9a591fb7-b88b-437c-8467-cf1e9b4b5a84',
 'd3b164c5-8347-4aec-99cf-52d0813fb4ae',
 '347729ce-08c0-4135-9d65-ea7be420ec6a',
 'acdc3493-a22d-405b-9a81-9f63ce3e6771',
 'f3f7dddb-d1de-48ea-83b3-05e769c12a40',
 '2d30f9e3-cc17-4788-a8b2-ee39220f489c',
 '5721515d-aadc-4b5e-896e-7f9782e4903e',
 '4ec419d8-3090-4367-8ead-925ea4921676',
 '662197bd-5407-4773-b47d-6647834e720a',
 'a5fbef45-118c-4a84-8559-848e7de08fa3',
 'ab8fcac9-418f-4076-bc30-32d7e0092fc1',
 'ef96884d-72ae-44f8-8032-af38c635b10b',
 'ec93ebbd-3135-4331-94f5-abf18115c38f',
 '45dfffa8-8052-4df9-8f48-b908b4346a3a',
 'a37ad3e5-e788-4e27-8094-948cf30fe7fb',
 'c5adf5f2-dc84-465b-8ba7-f95bb064cc27',
 '3cdec9cf-e861-4a47-800c-4df357d74038',
 '7c7afdbd-8280-4c5b-9569-8dce57b010da',
 'ddf0f6d2-67cf-4dab-a72c-1a482f477695',
 '45d4e1de-9c97-4305-a809-fd8765ad3b36',
 '8c56c616-d005-49cc-8545-f5258d2d3d29',
 '5413df43-a3ba-

### Sanity Check retrieval

The most complex table in the paper:

In [ ]:
tables

Here is the summary, which is embedded:

In [ ]:
table_summaries

[]

Here is our retrieval of that table from the natural language query:

In [ ]:
# We can retrieve this table
retriever.invoke("What are results for LLaMA across across domains / subjects?")

['LLaVA-Bench (In-the-Wild). To evaluate the model’s capability in more challenging tasks and generalizability to novel domains, we collect a diverse set of 24 images with 60 questions in total, including indoor and outdoor scenes, memes, paintings, sketches, etc., and associate each image with a highly-detailed and manually-curated description and a proper selection of questions. We compare LLaVA, BLIP, and OpenFlamingo in Table 5. Thanks to visual instruction tuning, LLaVA achieves significantly better performance compared with BLIP-2 (+29%) and OpenFlamingo (+48%). Compared to the text-only GPT-4 that has access to ground-truth labels, LLaVA achieves an impressive 81.7% performance on complex reasoning questions, with an overall score of 67.3%.\n\nLimitations. This LLaVA-Bench (In-the-Wild) is designed to be challenging and to reveal a model’s weaknesses. We provide two examples with associated captions and questions in Table 6. For the ramen example (left), to correctly answer the 

Image:

![figure-8-1.jpg](attachment:5d505f36-17e1-4fe5-a405-f01f7a392716.jpg)

We can retrieve this image summary:

In [ ]:
retriever.invoke("Images / figures with playful and creative examples")[1]

'3 GPT-assisted Visual Instruction Data Generation\n\nThe community has witnessed a surge in the amount of public multimodal data such as image-text pairs, ranging from CC [8] to LAION [45]. However, when it comes to multimodal instruction-\n\n2\n\nContext type 1: Captions\n\nA group of people standing outside of a black vehicle with various luggage.\n\nLuggage surrounds a vehicle in an underground parking area\n\nPeople try to fit all of their luggage in an SUV.\n\nThe sport utility vehicle is parked in the public garage, being packed for a trip\n\nSome people with luggage near a van that is transporting it.\n\nContext type 2: Boxes\n\nperson: [0.681, 0.242, 0.774, 0.694], backpack: [0.384, 0.696, 0.485, 0.914], suitcase: ...<omitted>'

## RAG

Run [RAG pipeline](https://python.langchain.com/docs/expression_language/cookbook/retrieval).

For `option 1` (above):

* Simply pass retrieved text chunks to LLM, as usual.

For `option 2a` (above):

* We would pass retrieved image and images to the multi-modal LLM.
* This should be possible soon, once [llama-cpp-python add multi-modal support](https://github.com/abetlen/llama-cpp-python/issues/813).
* And, of course, this will be enabled by GPT4-V API.

In [ ]:
from operator import itemgetter
from langchain_classic.schema.runnable import RunnablePassthrough

# Prompt template
template = """Answer the question based only on the following context, which can include text and tables:
{context}
Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# Option 1: LLM
model = ChatOpenAI(temperature=0,model="gpt-4")
# Option 2: Multi-modal LLM
# model = GPT4-V or LLaVA

# RAG pipeline
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:
chain.invoke("What is the performance of LLaVa across across multiple image domains / subjects?")

'LLaVA was evaluated on a diverse set of 24 images with 60 questions in total, including indoor and outdoor scenes, memes, paintings, sketches, etc. The model achieved significantly better performance compared with BLIP-2 (+29%) and OpenFlamingo (+48%). Compared to the text-only GPT-4 that has access to ground-truth labels, LLaVA achieves an impressive 81.7% performance on complex reasoning questions, with an overall score of 67.3%. However, the model did show some limitations, such as failing to correctly identify the presence of strawberry-flavored yogurt in an image of a fridge containing only yogurt and strawberries.'

We can check the [trace](https://smith.langchain.com/public/85a7180e-0dd1-44d9-996f-6cb9c6f53205/r) to see retrieval of tables and text.

In [ ]:
chain.invoke("Explain images / figures with playful and creative examples.")

"The text provides two examples of images that are explained in a playful and creative manner. \n\n1. The first example is a humorous artwork that mimics the famous painting, Mona Lisa. The artwork replaces the human figure with a dog dressed in a woman's clothing. LLaVA, the AI model, recognizes the humor in the artwork and explains that it is a creative and comical take on the traditional portrait style.\n\n2. The second example is a meme featuring Elon Musk dressed as a doge. Despite the humorous and unusual depiction, LLaVA is able to recognize Elon Musk in the image. This demonstrates the AI model's ability to generalize to unseen visual concepts and recognize familiar figures even in playful and creative contexts."